In [6]:
# Shared setup for CPU and FPGA benchmark cells
import numpy as np
import time

sizes = [32, 64, 128, 256, 512, 1024, 2048]
rng = np.random.default_rng(42)
test_matrices = {
    N: (
        rng.integers(-8, 8, size=(N, N), dtype=np.int16),
        rng.integers(-8, 8, size=(N, N), dtype=np.int16),
    )
    for N in sizes
}


In [7]:
# CPU benchmark
cpu_results = {}
cpu_warmup_runs = 3
cpu_repeats = 10

for N, (A, B) in test_matrices.items():
    A32 = A.astype(np.int32)
    B32 = B.astype(np.int32)

    for _ in range(cpu_warmup_runs):
        _ = A32 @ B32

    times = []
    C_cpu = None
    for _ in range(cpu_repeats):
        t0 = time.perf_counter()
        C_cpu = A32 @ B32
        t1 = time.perf_counter()
        times.append(t1 - t0)

    avg_t = float(np.mean(times))
    median_t = float(np.median(times))
    cpu_results[N] = {
        "matrix": C_cpu,
        "time": median_t,
        "avg_time": avg_t,
        "median_time": median_t,
        "all_times": times,
    }

    print(f"CPU  N={N:4d}  Avg={avg_t * 1e3:8.3f} ms  Median={median_t * 1e3:8.3f} ms")


CPU  N=  32  Avg=   0.024 ms  Median=   0.023 ms
CPU  N=  64  Avg=   0.149 ms  Median=   0.145 ms
CPU  N= 128  Avg=   1.344 ms  Median=   1.284 ms
CPU  N= 256  Avg=  10.089 ms  Median=  10.063 ms
CPU  N= 512  Avg= 170.916 ms  Median= 169.065 ms
CPU  N=1024  Avg=2513.741 ms  Median=2486.709 ms
CPU  N=2048  Avg=66140.140 ms  Median=66305.848 ms


In [ ]:
# FPGA benchmark
from pynq import Overlay, allocate

ol = Overlay("matmul.bit", download=True)
dma = ol.matmul_hier.axi_dma
matmul = ol.matmul_hier.matmul_bram_axis

def fpga_matmul_time(A, B):
    N = A.shape[0]
    in_buf = allocate(shape=(2 * N * N,), dtype=np.uint32)
    out_buf = allocate(shape=(N * N,), dtype=np.int32)

    in_buf[:N * N] = A.reshape(-1).astype(np.uint16).astype(np.uint32)
    in_buf[N * N:] = B.reshape(-1).astype(np.uint16).astype(np.uint32)
    out_buf[:] = 0

    matmul.write(0x10, N)     # size
    matmul.write(0x00, 0x01)  # ap_start

    t0 = time.perf_counter()
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.transfer(in_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    t1 = time.perf_counter()

    C_fpga = np.array(out_buf, dtype=np.int32).reshape(N, N)

    in_buf.freebuffer()
    out_buf.freebuffer()
    return C_fpga, (t1 - t0)

fpga_results = {}

for N, (A, B) in test_matrices.items():
    C_fpga, fpga_t = fpga_matmul_time(A, B)
    fpga_results[N] = {"matrix": C_fpga, "time": fpga_t}

    if N in cpu_results:
        ok = np.array_equal(cpu_results[N]["matrix"], C_fpga)
        speedup = cpu_results[N]["time"] / fpga_t if fpga_t > 0 else float("inf")
        print(f"FPGA N={N:4d}  OK={ok}  Time={fpga_t * 1e3:8.3f} ms  Speedup={speedup:6.2f}x")
    else:
        print(f"FPGA N={N:4d}  Time={fpga_t * 1e3:8.3f} ms")


In [ ]:
# assumes:
# dma = ol.matmul_hier.axi_dma
# matmul = ol.matmul_hier.matmul_bram_axis
# accelerator supports max block size = 256

BLOCK = 256

def fpga_matmul_block(A_blk, B_blk, dma, matmul):
    n = A_blk.shape[0]
    in_buf = allocate(shape=(2*n*n,), dtype=np.uint32)
    out_buf = allocate(shape=(n*n,), dtype=np.int32)

    in_buf[:n*n] = A_blk.reshape(-1).astype(np.uint16).astype(np.uint32)
    in_buf[n*n:] = B_blk.reshape(-1).astype(np.uint16).astype(np.uint32)
    out_buf[:] = 0

    matmul.write(0x10, n)     # size
    matmul.write(0x00, 0x01)  # ap_start

    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.transfer(in_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

    C_blk = np.array(out_buf, dtype=np.int32).reshape(n, n)

    in_buf.freebuffer()
    out_buf.freebuffer()
    return C_blk

def fpga_matmul_large(A, B, dma, matmul, block=BLOCK):
    N = A.shape[0]
    assert A.shape == (N, N) and B.shape == (N, N)
    assert N % block == 0, "For simplicity, use N multiple of block (e.g., 512, 1024)."

    C = np.zeros((N, N), dtype=np.int32)

    for i in range(0, N, block):
        for j in range(0, N, block):
            acc = np.zeros((block, block), dtype=np.int32)
            for k in range(0, N, block):
                A_blk = A[i:i+block, k:k+block].astype(np.int16)
                B_blk = B[k:k+block, j:j+block].astype(np.int16)
                acc += fpga_matmul_block(A_blk, B_blk, dma, matmul)
            C[i:i+block, j:j+block] = acc
    return C

# Example for 512 / 1024
for N in [512, 1024, 2048]:
    rng = np.random.default_rng(42)
    A = rng.integers(-8, 8, size=(N, N), dtype=np.int16)
    B = rng.integers(-8, 8, size=(N, N), dtype=np.int16)

    # CPU time
    t0 = time.perf_counter()
    C_cpu = A.astype(np.int32) @ B.astype(np.int32)
    t1 = time.perf_counter()
    cpu_ms = (t1 - t0) * 1000

    # FPGA time (blocked)
    t2 = time.perf_counter()
    C_fpga = fpga_matmul_large(A, B, dma, matmul, block=256)
    t3 = time.perf_counter()
    fpga_ms = (t3 - t2) * 1000

    ok = np.array_equal(C_fpga, C_cpu)
    speedup = cpu_ms / fpga_ms

    print(f"N={N:4d}  OK={ok}  CPU={cpu_ms:9.3f} ms  FPGA={fpga_ms:9.3f} ms  Speedup={speedup:6.2f}x")